In [2]:
%run start.py

Root set to: /home/bdudas/obesity_challange


In [3]:
import torch
import numpy as np
from src.data.vae_data import get_loaders
from omegaconf import OmegaConf
from src.models.transformerVAE import TransformerVAEEncoder, TransformerVAEDecoder,Transfomer_latent_Classifier


from src.models.vae_trainers import StateTrainer_latent

/home/bdudas/anaconda3/envs/vcell/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
z_dim = 128
traning_config = OmegaConf.load("configs/traning.yaml")
encoder_config = OmegaConf.load("configs/encoder.yaml")
encoder_config.z_dim = z_dim

decoder_config = OmegaConf.load("configs/decoder.yaml")
decoder_config.z_dim = z_dim

classifier_config = OmegaConf.load("configs/classifier_latent.yaml")
classifier_config.z_dim = z_dim
trainer_config = OmegaConf.load("configs/trainer.yaml")

In [5]:
train_loader,valloader, *_ = get_loaders(path = "",batch_size=128)

In [6]:
encoder = TransformerVAEEncoder(**encoder_config)
decoder = TransformerVAEDecoder(**decoder_config)
classifier = Transfomer_latent_Classifier(**classifier_config)

model = StateTrainer_latent(encoder,decoder,categorizer=classifier,**trainer_config)

/home/bdudas/anaconda3/envs/vcell/lib/python3.10/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


In [7]:
cpkt_path = "misc/best_runs/latent_reg/checkpoints/epoch=19-step=5520.ckpt"

state_dict = torch.load(cpkt_path,weights_only=False)

In [8]:
## Get only encoder weights

encoder_weights = {}
for key in state_dict['state_dict']:
    if key.startswith('encoder'):
        encoder_weights[key.replace('encoder.','')] = state_dict['state_dict'][key]

In [9]:
encoder_weights2 = {}
for k1, k2 in zip(encoder.state_dict().keys(), encoder_weights.keys()):
    if k1 == k2:
        encoder_weights2[k1] = encoder_weights[k2]
    else:
        encoder_weights2[k1] = encoder_weights[k2.replace('encoder.','')]

In [11]:
del state_dict

In [10]:
encoder.load_state_dict(encoder_weights2)

<All keys matched successfully>

# Generate low dimensional data

In [12]:
def reparameterize(mu, logvar):
    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(std)
    return mu + eps * std

In [13]:
batch = next(iter(valloader))
X, _, state = batch
mu, logvar = encoder(X)
logvar = torch.clamp(logvar, min=-6., max=2.)
z = reparameterize(mu, logvar)

In [18]:
sample_num = 10
manysample = reparameterize(mu.repeat(sample_num,1), logvar.repeat(sample_num,1))

In [21]:
manysample.shape

torch.Size([1280, 128])